<a href="https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Search Intelligence Data Contract

This notebook defines and verifies the data contract for my content refresh lane using the FlyRank warehouse.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


In [ ]:
REL = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Warehouse tables configured.")

Warehouse tables configured.


In [ ]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## My answer

For my lane, one row represents the observed search performance of one webpage for one client on one reporting date.

I will use March 2026 as my working time window. I chose March because it is a middle month in the available warehouse data, rather than the final June 2026 month.

The purpose is to use historical search-performance information to identify webpages that may deserve a content refresh.

In [ ]:
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS rows_per_key
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
ORDER BY rows_per_key DESC
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,rows_per_key
0,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,1
2,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,1
3,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,1
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,1
5,client_73cda7b4e4f265ea,content_2662845f598544ef,2026-03-01,1
6,client_73cda7b4e4f265ea,content_1855a661b4d36130,2026-03-01,1
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,2026-03-01,1
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,2026-03-01,1
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,2026-03-01,1


## 2. Fields: feature / label / context / excluded

### Features

I plan to use five historical signals:

1. Previous 30-day impressions
2. Previous 30-day clicks
3. Previous 30-day average position
4. Visible query count
5. Top query share

These features describe the page's historical search performance and query mix before the decision moment.

### Label

My label will be is_declining_label, which indicates whether the page's trend direction is "down".

### Context

- `client_hash_id` — identifies the client without exposing the client name.
- `content_hash_id` — identifies the content item.
- `report_date` — identifies the reporting date.

These fields help define and group the observations but are not intended to be predictive features.

### Excluded

I will exclude `trend_direction` and `trend_pct` from the feature set because the label is derived from `trend_direction`. Using either field as a feature would leak outcome information into the model.

In [ ]:
feature_names = [
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share",
]

label_name = "is_declining"

context_fields = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
]

excluded_fields = [
    "future_outcome_information",
]

print("Features:", feature_names)
print("Label:", label_name)
print("Context:", context_fields)
print("Excluded:", excluded_fields)

Features: ['previous_30d_impressions', 'previous_30d_clicks', 'previous_30d_avg_position', 'visible_query_count', 'top_query_share']
Label: is_declining
Context: ['client_hash_id', 'content_hash_id', 'report_date']
Excluded: ['future_outcome_information']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS gsc_available_rows
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
  AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_available_rows
0,3611061


## Five features

I will use five features for my content-refresh lane:

1. **Previous 30-day impressions** — knowable at the decision moment because these impressions were recorded before the decision window.
2. **Previous 30-day clicks** — knowable at the decision moment because these clicks come from historical Search Console data.
3. **Previous 30-day average position** — knowable at the decision moment because it is calculated from historical ranking observations.
4. **Visible query count** — knowable at the decision moment because it comes from the historical query-level data.
5. **Top query share** — knowable at the decision moment because it is calculated from the historical distribution of impressions across queries.

These features are intended to support prioritisation of pages for review, not to prove that a refresh will cause a traffic improvement.

In [ ]:
features = con.sql(f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-04-01'
),

march_pages AS (
    SELECT DISTINCT
        client_hash_id,
        content_hash_id
    FROM daily
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
),

feature_frame AS (
    SELECT
        m.client_hash_id,
        m.content_hash_id,

        SUM(
            CASE
                WHEN d.report_date >= DATE '2026-02-01'
                 AND d.report_date < DATE '2026-03-01'
                THEN COALESCE(d.gsc_impressions, 0)
                ELSE 0
            END
        ) AS previous_30d_impressions,

        SUM(
            CASE
                WHEN d.report_date >= DATE '2026-02-01'
                 AND d.report_date < DATE '2026-03-01'
                THEN COALESCE(d.gsc_clicks, 0)
                ELSE 0
            END
        ) AS previous_30d_clicks,

        AVG(
            CASE
                WHEN d.report_date >= DATE '2026-02-01'
                 AND d.report_date < DATE '2026-03-01'
                THEN d.gsc_avg_position
            END
        ) AS previous_30d_avg_position

    FROM march_pages m
    LEFT JOIN daily d
      ON m.client_hash_id = d.client_hash_id
     AND m.content_hash_id = d.content_hash_id

    GROUP BY
        m.client_hash_id,
        m.content_hash_id
)

SELECT *
FROM feature_frame
LIMIT 20
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,29.609070
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.806923
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,7.852945
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,7.123694
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,7.965842


In [ ]:
qsignals = con.sql(f"""
SELECT
    content_hash_id,
    ANY_VALUE(content_visible_query_count) AS visible_query_count,
    MAX(impressions_90d) AS top_query_impressions,
    SUM(impressions_90d) AS total_query_impressions
FROM {TABLES['fact_query_90d']}
GROUP BY content_hash_id
""").df()

qsignals["top_query_share"] = (
    qsignals["top_query_impressions"]
    / qsignals["total_query_impressions"]
)

print(qsignals.columns.tolist())
qsignals.head()

['content_hash_id', 'visible_query_count', 'top_query_impressions', 'total_query_impressions', 'top_query_share']


,content_hash_id,visible_query_count,top_query_impressions,total_query_impressions,top_query_share
0,content_447894f2faf0d2bc,14,55,339.0,0.162242
1,content_4486e5efcc7b773f,10,169,486.0,0.347737
2,content_448b01e1de5750b4,3,23,48.0,0.479167
3,content_44a3108ea1a95d57,5,235,524.0,0.448473
4,content_44c082bdb9a864ea,23,542,3242.0,0.167181


In [ ]:
features = features.merge(
    qsignals[
        [
            "content_hash_id",
            "visible_query_count",
            "top_query_share"
        ]
    ],
    on="content_hash_id",
    how="left"
)

print(features.columns.tolist())

['client_hash_id', 'content_hash_id', 'previous_30d_impressions', 'previous_30d_clicks', 'previous_30d_avg_position', 'visible_query_count', 'top_query_share']


In [ ]:
features.head()

,client_hash_id,content_hash_id,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position,visible_query_count,top_query_share
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,29.609070,25,0.260684
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.806923,21,0.111264
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,7.852945,5,0.606250
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,7.123694,3,0.371429
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,7.965842,4,0.568627


Part A — Leakage trap

In [ ]:
# Create the observed March outcome used as the label
outcomes = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(
        CASE
            WHEN report_date >= DATE '2026-03-01'
             AND report_date < DATE '2026-04-01'
            THEN COALESCE(gsc_impressions, 0)
            ELSE 0
        END
    ) AS march_impressions

FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

features = features.merge(
    outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

features["is_declining_label"] = (
    features["march_impressions"]
    < 0.8 * features["previous_30d_impressions"]
).astype(int)

features[[
    "previous_30d_impressions",
    "march_impressions",
    "is_declining_label"
]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,previous_30d_impressions,march_impressions,is_declining_label
0,1012.0,768.0,1
1,1598.0,3071.0,0
2,861.0,547.0,1
3,245.0,209.0,0
4,306.0,500.0,0


Part B — Deliberately create leakage

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

leak_data = features.dropna(
    subset=[
        "previous_30d_impressions",
        "previous_30d_clicks",
        "previous_30d_avg_position",
        "visible_query_count",
        "top_query_share",
        "is_declining_label"
    ]
).copy()

leak_features = [
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share",

    # DELIBERATE LEAK
    "is_declining_label"
]

X = leak_data[leak_features]
y = leak_data["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leak_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leak_model.fit(X_train, y_train)

leak_score = accuracy_score(
    y_test,
    leak_model.predict(X_test)
)

print("Deliberate leakage accuracy:", round(leak_score, 3))

Deliberate leakage accuracy: 1.0


### Leakage experiment

I deliberately added `is_declining_label` as a feature to demonstrate leakage.

The score became extremely high because the model was given information that directly represents the outcome it was supposed to predict.

This result is not useful evidence of model quality. It shows why label-derived or future outcome information must be excluded from the honest feature set.

The five historical features are kept as the honest feature set because they are intended to be available before the decision moment.

Part D — Remove the leakage

In [ ]:
honest_features = [
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share"
]

print("Honest features:")
print(honest_features)

print("\nLeakage feature removed:")
print("is_declining_label")

Honest features:
['previous_30d_impressions', 'previous_30d_clicks', 'previous_30d_avg_position', 'visible_query_count', 'top_query_share']

Leakage feature removed:
is_declining_label


## 4. Data limits

This data can support observed and directional analysis, but it cannot tell me that a content refresh will definitely cause traffic to improve.

The warehouse has an unbalanced history, meaning different clients have different amounts of historical data. This makes comparisons across clients less uniform.

Some early rows can have GSC data available while GA4 data is not yet available, so missing data should not automatically be treated as zero performance.

My feature and outcome windows also need to stay separate. If the feature window overlaps the outcome window, the model can see information from the future and produce misleadingly strong results.

The data also cannot prove Google's ranking algorithm or establish causality without an appropriate experiment or causal design.

Therefore, my output should be treated as decision-support for prioritising pages for review, not as a guarantee that a page will recover after a refresh.## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
print("Named limitation:")
print("The warehouse has unbalanced client histories, so not every client has the same amount of historical data.")

print("\nDecision-support limitation:")
print("The analysis can identify and prioritise pages for review, but it cannot prove that a refresh will cause recovery.")

Named limitation:
The warehouse has unbalanced client histories, so not every client has the same amount of historical data.

Decision-support limitation:
The analysis can identify and prioritise pages for review, but it cannot prove that a refresh will cause recovery.


## Self-check

Before you submit, confirm each line honestly:

☑ Every section above is filled — markdown thinking AND code

☑ The notebook runs top to bottom with no errors

☑ No client names, URLs, or private queries

☑ Claims use careful words: observed, measured, directional, decision-support

☑ Committed to my repo under work/notebooks/